#  CardioIA — Fase 2: Classificador de Risco Cardíaco

**Projeto:** PBL CardioIA - A Nova Era da Cardiologia Inteligente  
**Aluno:** Lucas Yuji Nakayama Hirano — RM 563420  

---

## Objetivo
Nesta Parte 2, construímos um classificador automático de risco cardíaco utilizando:
- **TF-IDF** para vetorizar as frases dos pacientes (transformar texto em números)
- **Regressão Logística** para classificar o nível de risco: `alto risco` ou `baixo risco`

>  *Este projeto é uma simulação educacional. Não deve ser usado para diagnósticos reais.*

---
##  Etapa 1 — Importação das Bibliotecas

In [34]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

print(' Bibliotecas importadas com sucesso!')

 Bibliotecas importadas com sucesso!


---
##  Etapa 2 — Carregamento e Visualização dos Dados

Carregamos o arquivo `dataset_risco.csv`, que contém frases de pacientes já rotuladas como `alto risco` ou `baixo risco`.

In [33]:
# Carregar o dataset
df = pd.read_csv('dataset_risco.csv', sep=',', encoding='utf-8')

# Verificar se as colunas foram carregadas corretamente
if 'frase,situacao' in df.columns:
	# Se não foi parseado corretamente, tentar novamente
	df = pd.read_csv('dataset_risco.csv', encoding='utf-8')
	# Separar a coluna que contém ambos os valores
	df[['frase', 'situacao']] = df['frase,situacao'].str.split(',', n=1, expand=True)
	df = df[['frase', 'situacao']]

print(f' Total de registros carregados: {len(df)}')
print(f'\nDistribuição das classes:')
print(df['situacao'].value_counts())
print('\n--- Primeiros registros ---')
df.head(12)

 Total de registros carregados: 12

Distribuição das classes:
situacao
alto risco     7
baixo risco    5
Name: count, dtype: int64

--- Primeiros registros ---


,frase,situacao
0,sinto dor no peito e falta de ar,alto risco
1,tive um leve incômodo nas costas,baixo risco
2,dor forte no peito que irradia para o braço es...,alto risco
3,cansaço constante mesmo após descansar,alto risco
4,leve dor de cabeça após o trabalho,baixo risco
5,palpitações frequentes e suor frio,alto risco
6,só sinto um desconforto leve no peito ao subir...,baixo risco
7,falta de ar intensa e inchaço nas pernas,alto risco
8,tive azia depois do almoço,baixo risco
9,aperto no tórax com tontura e suor frio,alto risco


---
##  Etapa 3 — Vetorização com TF-IDF

**TF-IDF** (Term Frequency–Inverse Document Frequency) transforma frases de texto em vetores numéricos.  
Palavras que aparecem muito em um documento mas pouco nos outros recebem maior peso — ou seja, são mais relevantes.

- **TF (Term Frequency):** frequência de uma palavra na frase
- **IDF (Inverse Document Frequency):** penaliza palavras muito comuns em todos os documentos

Isso permite que o modelo de ML "entenda" o texto.

In [35]:
# Separar features (X) e rótulos (y)
X = df['frase']
y = df['situacao']

# Criar e aplicar o vetorizador TF-IDF
# max_features=100 significa que usaremos no máximo 100 palavras mais relevantes
vectorizer = TfidfVectorizer(max_features=100)
X_vec = vectorizer.fit_transform(X)

print(f' Vetorização concluída!')
print(f'   Shape da matriz TF-IDF: {X_vec.shape}')
print(f'   (linhas = frases, colunas = palavras únicas encontradas)')
print(f'\n Palavras (features) extraídas pelo TF-IDF:')
print(vectorizer.get_feature_names_out())

 Vetorização concluída!
   Shape da matriz TF-IDF: (12, 51)
   (linhas = frases, colunas = palavras únicas encontradas)

 Palavras (features) extraídas pelo TF-IDF:
['almoço' 'ao' 'aperto' 'após' 'ar' 'azia' 'braço' 'cabeça' 'cansaço'
 'com' 'constante' 'costas' 'de' 'depois' 'descansar' 'desconforto' 'do'
 'dor' 'escadas' 'esforço' 'esquerdo' 'fadiga' 'falta' 'forte'
 'frequentes' 'frio' 'inchaço' 'incômodo' 'intensa' 'irradia' 'leve'
 'mesmo' 'nas' 'no' 'palpitações' 'para' 'passa' 'peito' 'pernas' 'piora'
 'que' 'repouso' 'sinto' 'subir' 'suor' 'só' 'tive' 'tontura' 'trabalho'
 'tórax' 'um']


---
##  Etapa 4 — Divisão Treino / Teste

Dividimos os dados em dois conjuntos:
- **Treino (70%):** o modelo aprende os padrões
- **Teste (30%):** avaliamos o desempenho em dados que o modelo nunca viu

O `random_state=42` garante que a divisão seja sempre a mesma (reprodutibilidade).

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.3, random_state=42
)

print(f'📚 Dados de treino:  {X_train.shape[0]} frases ({100 - 30}%)')
print(f'🧪 Dados de teste:   {X_test.shape[0]} frases (30%)')

📚 Dados de treino:  8 frases (70%)
🧪 Dados de teste:   4 frases (30%)


---
##  Etapa 5 — Treinamento do Modelo (Regressão Logística)

A **Regressão Logística** é um algoritmo de classificação que aprende a separar classes (aqui: `alto risco` vs `baixo risco`) traçando uma fronteira de decisão baseada nos pesos de cada palavra.  
Apesar do nome "regressão", é um classificador — muito usado em NLP por ser rápido e interpretável.

In [ ]:
# Criar e treinar o modelo
modelo = LogisticRegression(random_state=42, max_iter=200)
modelo.fit(X_train, y_train)

print(' Modelo treinado com sucesso!')

✅ Modelo treinado com sucesso!


---
##  Etapa 6 — Avaliação do Modelo

Avaliamos o modelo com métricas padrão de classificação:

| Métrica | O que mede |
|---|---|
| **Acurácia** | % de previsões corretas no total |
| **Precisão** | Dos que o modelo classificou como alto risco, quantos realmente eram? |
| **Recall** | Dos que realmente eram alto risco, quantos o modelo encontrou? |
| **F1-Score** | Média harmônica entre precisão e recall |

In [36]:
# Fazer previsões no conjunto de teste
y_pred = modelo.predict(X_test)

# Calcular métricas
acuracia = accuracy_score(y_test, y_pred)

print('=' * 45)
print('       AVALIAÇÃO DO MODELO CARDIOIA')
print('=' * 45)
print(f'\n Acurácia: {acuracia * 100:.1f}%\n')
print(' Relatório completo:')
print(classification_report(y_test, y_pred))

       AVALIAÇÃO DO MODELO CARDIOIA

 Acurácia: 50.0%

 Relatório completo:
              precision    recall  f1-score   support

  alto risco       0.50      1.00      0.67         2
 baixo risco       0.00      0.00      0.00         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4



c:\Users\yuji_\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yuji_\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yuji_\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

---
##  Etapa 7 — Teste com Novas Frases

Simulamos o uso real do modelo: enviamos frases novas (que o modelo nunca viu) e ele classifica o risco de cada paciente.

In [30]:
frases_novas = [
    "dor no peito irradiando para o braço e falta de ar",
    "só um leve cansaço após caminhar",
    "palpitações fortes com suor frio e tontura",
    "sinto um leve desconforto que passa sozinho",
    "falta de ar intensa com inchaço nas pernas"
]

print('=' * 55)
print('      CLASSIFICAÇÃO DE RISCO — NOVOS PACIENTES')
print('=' * 55)

for i, frase in enumerate(frases_novas, 1):
    vec = vectorizer.transform([frase])
    pred = modelo.predict(vec)[0]
    prob = modelo.predict_proba(vec)[0]
    confianca = max(prob) * 100
    emoji = '🔴' if pred == 'alto risco' else '🟢'
    print(f'\nPaciente {i}: "{frase}"')
    print(f'  {emoji} Risco: {pred.upper()} (confiança: {confianca:.1f}%)')

      CLASSIFICAÇÃO DE RISCO — NOVOS PACIENTES

Paciente 1: "dor no peito irradiando para o braço e falta de ar"
  🔴 Risco: ALTO RISCO (confiança: 69.1%)

Paciente 2: "só um leve cansaço após caminhar"
  🔴 Risco: ALTO RISCO (confiança: 54.8%)

Paciente 3: "palpitações fortes com suor frio e tontura"
  🔴 Risco: ALTO RISCO (confiança: 68.8%)

Paciente 4: "sinto um leve desconforto que passa sozinho"
  🔴 Risco: ALTO RISCO (confiança: 54.8%)

Paciente 5: "falta de ar intensa com inchaço nas pernas"
  🔴 Risco: ALTO RISCO (confiança: 69.8%)


---
##  Etapa 8 — Salvar Modelo e Vectorizer

Salvamos o modelo treinado e o vectorizer em arquivos `.pkl` (pickle).  
Isso permite reutilizá-los futuramente sem precisar treinar novamente — simulando um sistema de produção.

In [37]:
joblib.dump(modelo, 'modelo_risco.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

print(' Arquivos salvos com sucesso!')
print('   → modelo_risco.pkl')
print('   → vectorizer.pkl')

 Arquivos salvos com sucesso!
   → modelo_risco.pkl
   → vectorizer.pkl


---
##  Etapa 9 (Bônus) — Carregar e Reutilizar o Modelo Salvo

Demonstramos como carregar o modelo salvo e usá-lo para novas previsões — sem retreinar.

In [39]:
# Carregar modelo e vectorizer do disco
modelo_carregado = joblib.load('modelo_risco.pkl')
vectorizer_carregado = joblib.load('vectorizer.pkl')

# Testar com uma frase nova
frase_final = "aperto no peito com suor frio e dificuldade para respirar"
vec_final = vectorizer_carregado.transform([frase_final])
resultado = modelo_carregado.predict(vec_final)[0]

print(' Modelo carregado do disco com sucesso!')
print(f'\n🩺 Frase: "{frase_final}"')
print(f'   Diagnóstico de risco: {resultado.upper()}')

 Modelo carregado do disco com sucesso!

🩺 Frase: "aperto no peito com suor frio e dificuldade para respirar"
   Diagnóstico de risco: ALTO RISCO


---
## Conclusão

Nesta Parte 2 do projeto CardioIA, construímos com sucesso um classificador de risco cardíaco que:

1. **Carregou** e explorou o dataset de frases rotuladas
2. **Vetorizou** o texto usando TF-IDF
3. **Treinou** um modelo de Regressão Logística
4. **Avaliou** o desempenho com acurácia, precisão, recall e F1-Score
5. **Classificou** novas frases com o nível de risco e percentual de confiança
6. **Salvou e recarregou** o modelo para uso futuro

---

###  Considerações sobre IA Responsável

- Este projeto usa **dados sintéticos** para fins educacionais
- Em sistemas reais, o modelo precisaria de **validação clínica rigorosa** por médicos
- É fundamental respeitar a **privacidade do paciente (LGPD)** e aplicar **governança de dados**
- Vieses nos dados de treino podem levar a **diagnósticos incorretos** — daí a importância da diversidade e representatividade dos dados

---
*Lucas Yuji Nakayama Hirano — RM 563420*